# 1 Импорт библиотек и функций

## 1.1 Библиотеки

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings('ignore')

## 1.2 Функции

In [5]:
def evaluate_classification(y_test, y_pred_proba):
    """Оценивает результаты классификации"""
    # Кодируем строковые метки в числа
    le = LabelEncoder()
    y_test_encoded = le.fit_transform(y_test)
    
    # Предсказанные классы
    y_pred = np.argmax(y_pred_proba, axis=1)
    
    # Метрики
    accuracy = accuracy_score(y_test_encoded, y_pred)
    f1 = f1_score(y_test_encoded, y_pred, average='weighted')
    
    # Для многоклассовой классификации
    auc_roc = roc_auc_score(y_test_encoded, y_pred_proba, multi_class='ovr', average='weighted')
    
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1-score: {f1:.4f}")
    print(f"AUC-ROC: {auc_roc:.4f}")
    
    print(f"\nClassification Report:")
    print(classification_report(y_test_encoded, y_pred, target_names=['L (падение)', 'N (нейтр)', 'R (рост)']))

In [8]:
def train_predict_rf(X_train, X_test, y_train):
    """Обучает и предсказывает Random Forest"""
    le = LabelEncoder()
    y_train_encoded = le.fit_transform(y_train)
    
    model = RandomForestClassifier(n_estimators=100, random_state=13)
    model.fit(X_train, y_train_encoded)
    y_pred_proba = model.predict_proba(X_test)
    return y_pred_proba

def train_predict_dt(X_train, X_test, y_train):
    """Обучает и предсказывает Decision Tree"""
    le = LabelEncoder()
    y_train_encoded = le.fit_transform(y_train)
    
    model = DecisionTreeClassifier(random_state=13)
    model.fit(X_train, y_train_encoded)
    y_pred_proba = model.predict_proba(X_test)
    return y_pred_proba

def train_predict_logreg(X_train, X_test, y_train):
    """Обучает и предсказывает Logistic Regression"""
    le = LabelEncoder()
    y_train_encoded = le.fit_transform(y_train)
    
    model = LogisticRegression(max_iter=1000, random_state=13)
    model.fit(X_train, y_train_encoded)
    y_pred_proba = model.predict_proba(X_test)
    return y_pred_proba

def train_predict_catboost(X_train, X_test, y_train):
    """Обучает и предсказывает CatBoost"""
    le = LabelEncoder()
    y_train_encoded = le.fit_transform(y_train)
    
    model = CatBoostClassifier(verbose=False, random_state=13)
    model.fit(X_train, y_train_encoded)
    y_pred_proba = model.predict_proba(X_test)
    return y_pred_proba

In [10]:
def train_test_split_by_date(df, target_column, test_size=0.2):
    """
    Разбивает данные на train/test по дате и возвращает X, y
    
    Args:
        df: DataFrame с колонкой 'begin'
        target_column: название целевой переменной
        test_size: доля тестовых данных (0.2 = 20%)
    """
    df = df.sort_values('begin').reset_index(drop=True)
    
    # Вычисляем индекс разбиения
    split_idx = int(len(df) * (1 - test_size))
    
    train_df = df.iloc[:split_idx].copy()
    test_df = df.iloc[split_idx:].copy()
    
    print(f"Train: {train_df['begin'].min()} - {train_df['begin'].max()} ({len(train_df)} samples)")
    print(f"Test:  {test_df['begin'].min()} - {test_df['begin'].max()} ({len(test_df)} samples)")
    
    # Удаляем колонку 'begin' и разделяем на X, y
    X_train = train_df.drop(columns=['begin', target_column])
    y_train = train_df[target_column]
    
    X_test = test_df.drop(columns=['begin', target_column])
    y_test = test_df[target_column]
    
    print(f"Признаков: {X_train.shape[1]}")
    
    return X_train, X_test, y_train, y_test

# 2 Подготовка данных

## 2.0 Список тикеров

In [14]:
tickers = [
    'SBER', 'TCSG', 'GAZP', 'LKOH', 'ROSN'
]

## 2.1 Чтение

In [17]:
data_SBER = pd.read_csv("../../data/stock_features_data/stocks_features_SBER.csv")
data_TCSG = pd.read_csv("../../data/stock_features_data/stocks_features_TCSG.csv")
data_GAZP = pd.read_csv("../../data/stock_features_data/stocks_features_GAZP.csv")
data_LKOH = pd.read_csv("../../data/stock_features_data/stocks_features_LKOH.csv")
data_ROSN = pd.read_csv("../../data/stock_features_data/stocks_features_ROSN.csv")
data_ROSN.head(2)

,begin,close,MA_90,RSI_14,RSI_90,MOM_10,ATR_14,VOLATILITY_20,VOLATILITY_50,VOLUME_RATIO_20,MACD_SIGNAL,MACD_HISTOGRAM,target_price_change,target_class
0,2022-05-31 18:00:00,377.00,385.952778,41.700213,46.109636,-6.102117,6.815141,0.177819,0.184228,0.180549,1.622548,-2.115281,-4.721485,L
1,2022-06-01 10:00:00,378.35,385.708333,43.536971,46.522581,-3.703232,6.960488,0.153935,0.177697,0.770980,1.125999,-1.986196,-4.916083,L


## 2.2 Удаление лишних признаков

In [20]:
# Для SBER
part_SBER = data_SBER[['begin', 'close', 'target_price_change']]
data_SBER.drop(['close', 'target_price_change'], axis=1, inplace=True)

# Для TCSG
part_TCSG = data_TCSG[['begin', 'close', 'target_price_change']]
data_TCSG.drop(['close', 'target_price_change'], axis=1, inplace=True)

# Для GAZP
part_GAZP = data_GAZP[['begin', 'close', 'target_price_change']]
data_GAZP.drop(['close', 'target_price_change'], axis=1, inplace=True)

# Для LKOH
part_LKOH = data_LKOH[['begin', 'close', 'target_price_change']]
data_LKOH.drop(['close', 'target_price_change'], axis=1, inplace=True)

# Для ROSN
part_ROSN = data_ROSN[['begin', 'close', 'target_price_change']]
data_ROSN.drop(['close', 'target_price_change'], axis=1, inplace=True)

## 2.3 Разделение на train/test

In [23]:
X_train_SBER, X_test_SBER, y_train_SBER, y_test_SBER = train_test_split_by_date(
    df=data_SBER, 
    target_column='target_class',
    test_size=0.2
)

X_train_TCSG, X_test_TCSG, y_train_TCSG, y_test_TCSG = train_test_split_by_date(
    df=data_TCSG, 
    target_column='target_class',
    test_size=0.2
)

X_train_GAZP, X_test_GAZP, y_train_GAZP, y_test_GAZP = train_test_split_by_date(
    df=data_GAZP, 
    target_column='target_class',
    test_size=0.2
)

X_train_LKOH, X_test_LKOH, y_train_LKOH, y_test_LKOH = train_test_split_by_date(
    df=data_LKOH, 
    target_column='target_class',
    test_size=0.2
)

X_train_ROSN, X_test_ROSN, y_train_ROSN, y_test_ROSN = train_test_split_by_date(
    df=data_ROSN, 
    target_column='target_class',
    test_size=0.2
)

Train: 2022-05-31 18:00:00 - 2025-04-01 16:00:00 (3642 samples)
Test:  2025-04-01 18:00:00 - 2025-10-15 18:00:00 (911 samples)
Признаков: 10
Train: 2022-05-31 18:00:00 - 2024-06-06 10:00:00 (2472 samples)
Test:  2024-06-06 12:00:00 - 2024-11-27 18:00:00 (618 samples)
Признаков: 10
Train: 2022-05-31 18:00:00 - 2025-04-01 16:00:00 (3642 samples)
Test:  2025-04-01 18:00:00 - 2025-10-15 18:00:00 (911 samples)
Признаков: 10
Train: 2022-05-31 18:00:00 - 2025-04-01 16:00:00 (3642 samples)
Test:  2025-04-01 18:00:00 - 2025-10-15 18:00:00 (911 samples)
Признаков: 10
Train: 2022-05-31 18:00:00 - 2025-04-01 16:00:00 (3642 samples)
Test:  2025-04-01 18:00:00 - 2025-10-15 18:00:00 (911 samples)
Признаков: 10


# 3 Обучение простых моделей 

## 3.1 Дерево решений

### 3.1.1 Сбер

In [62]:
y_pred_SBER = train_predict_dt(X_train_SBER, X_test_SBER, y_train_SBER)
evaluate_classification(y_test_SBER, y_pred_SBER)

Accuracy: 0.7113
F1-score: 0.7169
AUC-ROC: 0.7872

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.80      0.78      0.79       376
   N (нейтр)       0.82      0.71      0.76       303
    R (рост)       0.50      0.59      0.54       232

    accuracy                           0.71       911
   macro avg       0.70      0.70      0.70       911
weighted avg       0.73      0.71      0.72       911



### 3.1.2 Тиньк

In [31]:
y_pred_TCSG = train_predict_dt(X_train_TCSG, X_test_TCSG, y_train_TCSG)
evaluate_classification(y_test_TCSG, y_pred_TCSG)

Accuracy: 0.7249
F1-score: 0.7217
AUC-ROC: 0.7870

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.78      0.82      0.80       285
   N (нейтр)       0.80      0.78      0.79       213
    R (рост)       0.43      0.39      0.41       120

    accuracy                           0.72       618
   macro avg       0.67      0.67      0.67       618
weighted avg       0.72      0.72      0.72       618



### 3.1.3 Газпром

In [34]:
y_pred_GAZP = train_predict_dt(X_train_GAZP, X_test_GAZP, y_train_GAZP)
evaluate_classification(y_test_GAZP, y_pred_GAZP)

Accuracy: 0.7135
F1-score: 0.7264
AUC-ROC: 0.7900

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.86      0.76      0.81       463
   N (нейтр)       0.75      0.77      0.76       323
    R (рост)       0.28      0.39      0.33       125

    accuracy                           0.71       911
   macro avg       0.63      0.64      0.63       911
weighted avg       0.75      0.71      0.73       911



### 3.1.4 Лукойл

In [37]:
y_pred_LKOH = train_predict_dt(X_train_LKOH, X_test_LKOH, y_train_LKOH)
evaluate_classification(y_test_LKOH, y_pred_LKOH)

Accuracy: 0.7080
F1-score: 0.7096
AUC-ROC: 0.7841

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.82      0.83      0.82       391
   N (нейтр)       0.77      0.73      0.75       295
    R (рост)       0.45      0.47      0.46       225

    accuracy                           0.71       911
   macro avg       0.68      0.68      0.68       911
weighted avg       0.71      0.71      0.71       911



### 3.1.5 Роснефть

In [40]:
y_pred_ROSN = train_predict_dt(X_train_ROSN, X_test_ROSN, y_train_ROSN)
evaluate_classification(y_test_ROSN, y_pred_ROSN)

Accuracy: 0.7333
F1-score: 0.7372
AUC-ROC: 0.8062

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.89      0.83      0.86       407
   N (нейтр)       0.74      0.77      0.75       320
    R (рост)       0.42      0.46      0.44       184

    accuracy                           0.73       911
   macro avg       0.68      0.69      0.68       911
weighted avg       0.74      0.73      0.74       911



## Вывод

Не плохие результаты для самой базовой модели ) </br>
На разных акциях точность отличается

## 3.2 Регрессия

### 3.2.1 Сбер

In [46]:
y_pred_SBER = train_predict_logreg(X_train_SBER, X_test_SBER, y_train_SBER)
evaluate_classification(y_test_SBER, y_pred_SBER)

Accuracy: 0.7124
F1-score: 0.7242
AUC-ROC: 0.9028

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.80      0.70      0.75       376
   N (нейтр)       0.90      0.77      0.83       303
    R (рост)       0.47      0.65      0.54       232

    accuracy                           0.71       911
   macro avg       0.72      0.71      0.71       911
weighted avg       0.75      0.71      0.72       911



### 3.2.2 Тиньк

In [49]:
y_pred_TCSG = train_predict_logreg(X_train_TCSG, X_test_TCSG, y_train_TCSG)
evaluate_classification(y_test_TCSG, y_pred_TCSG)

Accuracy: 0.7896
F1-score: 0.7848
AUC-ROC: 0.9369

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.90      0.87      0.88       285
   N (нейтр)       0.78      0.88      0.83       213
    R (рост)       0.51      0.43      0.47       120

    accuracy                           0.79       618
   macro avg       0.73      0.73      0.73       618
weighted avg       0.78      0.79      0.78       618



### 3.2.3 Газпром

In [52]:
y_pred_GAZP = train_predict_logreg(X_train_GAZP, X_test_GAZP, y_train_GAZP)
evaluate_classification(y_test_GAZP, y_pred_GAZP)

Accuracy: 0.7772
F1-score: 0.7850
AUC-ROC: 0.9297

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.88      0.84      0.86       463
   N (нейтр)       0.84      0.82      0.83       323
    R (рост)       0.35      0.44      0.39       125

    accuracy                           0.78       911
   macro avg       0.69      0.70      0.69       911
weighted avg       0.79      0.78      0.79       911



### 3.2.4 Лукойл

In [55]:
y_pred_LKOH = train_predict_logreg(X_train_LKOH, X_test_LKOH, y_train_LKOH)
evaluate_classification(y_test_LKOH, y_pred_LKOH)

Accuracy: 0.7629
F1-score: 0.7660
AUC-ROC: 0.9220

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.89      0.84      0.87       391
   N (нейтр)       0.79      0.80      0.80       295
    R (рост)       0.53      0.57      0.55       225

    accuracy                           0.76       911
   macro avg       0.74      0.74      0.74       911
weighted avg       0.77      0.76      0.77       911



### 3.2.5 Роснефть

In [58]:
y_pred_ROSN = train_predict_logreg(X_train_ROSN, X_test_ROSN, y_train_ROSN)
evaluate_classification(y_test_ROSN, y_pred_ROSN)

Accuracy: 0.7816
F1-score: 0.7845
AUC-ROC: 0.9314

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.91      0.86      0.89       407
   N (нейтр)       0.80      0.82      0.81       320
    R (рост)       0.50      0.53      0.52       184

    accuracy                           0.78       911
   macro avg       0.74      0.74      0.74       911
weighted avg       0.79      0.78      0.78       911



## Вывод

Качество лучше по сравнению с деревом

## 3.3 Вот он, лес

### 3.3.1 Сбер

In [66]:
y_pred_SBER = train_predict_rf(X_train_SBER, X_test_SBER, y_train_SBER)
evaluate_classification(y_test_SBER, y_pred_SBER)

Accuracy: 0.7420
F1-score: 0.7468
AUC-ROC: 0.9058

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.78      0.82      0.80       376
   N (нейтр)       0.91      0.76      0.83       303
    R (рост)       0.52      0.59      0.55       232

    accuracy                           0.74       911
   macro avg       0.74      0.72      0.73       911
weighted avg       0.76      0.74      0.75       911



### 3.3.2 Тиньк

In [69]:
y_pred_TCSG = train_predict_rf(X_train_TCSG, X_test_TCSG, y_train_TCSG)
evaluate_classification(y_test_TCSG, y_pred_TCSG)

Accuracy: 0.7799
F1-score: 0.7738
AUC-ROC: 0.9312

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.83      0.90      0.86       285
   N (нейтр)       0.85      0.83      0.84       213
    R (рост)       0.49      0.42      0.45       120

    accuracy                           0.78       618
   macro avg       0.72      0.71      0.72       618
weighted avg       0.77      0.78      0.77       618



### 3.3.3 Газпром

In [72]:
y_pred_GAZP = train_predict_rf(X_train_GAZP, X_test_GAZP, y_train_GAZP)
evaluate_classification(y_test_GAZP, y_pred_GAZP)

Accuracy: 0.7849
F1-score: 0.7931
AUC-ROC: 0.9378

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.89      0.85      0.87       463
   N (нейтр)       0.85      0.81      0.83       323
    R (рост)       0.37      0.47      0.42       125

    accuracy                           0.78       911
   macro avg       0.70      0.71      0.70       911
weighted avg       0.80      0.78      0.79       911



### 3.3.4 Лукойл

In [75]:
y_pred_LKOH = train_predict_rf(X_train_LKOH, X_test_LKOH, y_train_LKOH)
evaluate_classification(y_test_LKOH, y_pred_LKOH)

Accuracy: 0.7673
F1-score: 0.7649
AUC-ROC: 0.9268

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.83      0.89      0.86       391
   N (нейтр)       0.82      0.77      0.80       295
    R (рост)       0.57      0.55      0.56       225

    accuracy                           0.77       911
   macro avg       0.74      0.74      0.74       911
weighted avg       0.76      0.77      0.76       911



### 3.3.5 Роснефть

In [78]:
y_pred_ROSN = train_predict_rf(X_train_ROSN, X_test_ROSN, y_train_ROSN)
evaluate_classification(y_test_ROSN, y_pred_ROSN)

Accuracy: 0.7958
F1-score: 0.7948
AUC-ROC: 0.9299

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.88      0.89      0.89       407
   N (нейтр)       0.83      0.83      0.83       320
    R (рост)       0.54      0.52      0.53       184

    accuracy                           0.80       911
   macro avg       0.75      0.75      0.75       911
weighted avg       0.79      0.80      0.79       911



## Вывод

Показывает лучшее качество по сравнению с деревом

## 3.4 Бустинг

### 3.4.1 Сбер

In [83]:
y_pred_SBER = train_predict_catboost(X_train_SBER, X_test_SBER, y_train_SBER)
evaluate_classification(y_test_SBER, y_pred_SBER)

Accuracy: 0.7442
F1-score: 0.7496
AUC-ROC: 0.9065

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.78      0.81      0.80       376
   N (нейтр)       0.91      0.78      0.84       303
    R (рост)       0.53      0.60      0.56       232

    accuracy                           0.74       911
   macro avg       0.74      0.73      0.73       911
weighted avg       0.76      0.74      0.75       911



### 3.4.2 Тиньк

In [86]:
y_pred_TCSG = train_predict_catboost(X_train_TCSG, X_test_TCSG, y_train_TCSG)
evaluate_classification(y_test_TCSG, y_pred_TCSG)

Accuracy: 0.7832
F1-score: 0.7812
AUC-ROC: 0.9284

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.85      0.90      0.87       285
   N (нейтр)       0.85      0.81      0.83       213
    R (рост)       0.49      0.47      0.48       120

    accuracy                           0.78       618
   macro avg       0.73      0.72      0.73       618
weighted avg       0.78      0.78      0.78       618



### 3.4.3 Газпром

In [89]:
y_pred_GAZP = train_predict_catboost(X_train_GAZP, X_test_GAZP, y_train_GAZP)
evaluate_classification(y_test_GAZP, y_pred_GAZP)

Accuracy: 0.8068
F1-score: 0.8157
AUC-ROC: 0.9481

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.91      0.86      0.88       463
   N (нейтр)       0.88      0.84      0.86       323
    R (рост)       0.40      0.53      0.46       125

    accuracy                           0.81       911
   macro avg       0.73      0.74      0.73       911
weighted avg       0.83      0.81      0.82       911



### 3.4.4 Лукойл

In [92]:
y_pred_LKOH = train_predict_catboost(X_train_LKOH, X_test_LKOH, y_train_LKOH)
evaluate_classification(y_test_LKOH, y_pred_LKOH)

Accuracy: 0.7750
F1-score: 0.7783
AUC-ROC: 0.9324

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.87      0.87      0.87       391
   N (нейтр)       0.84      0.78      0.81       295
    R (рост)       0.55      0.61      0.58       225

    accuracy                           0.77       911
   macro avg       0.75      0.75      0.75       911
weighted avg       0.78      0.77      0.78       911



### 3.4.5 Роснефть

In [95]:
y_pred_ROSN = train_predict_catboost(X_train_ROSN, X_test_ROSN, y_train_ROSN)
evaluate_classification(y_test_ROSN, y_pred_ROSN)

Accuracy: 0.7859
F1-score: 0.7823
AUC-ROC: 0.9254

Classification Report:
              precision    recall  f1-score   support

 L (падение)       0.89      0.89      0.89       407
   N (нейтр)       0.80      0.84      0.82       320
    R (рост)       0.51      0.46      0.48       184

    accuracy                           0.79       911
   macro avg       0.73      0.73      0.73       911
weighted avg       0.78      0.79      0.78       911



## Вывод

Бустинг показал лучшее качество на всех акциях кроме Роснефти по сравнению с лесом, даже на Сбере )

В данном сравнении лучше всего показал себя градиентный бустинг